# Bốn vector store trên cùng 61 425 vector: FAISS, Chroma, Qdrant (local), LanceDB

Cùng một ma trận embedding `bge-m3` của Zalo Legal, bốn công cụ, một giao diện chung. Bạn viết adapter cho ba công cụ
(Chroma có sẵn làm mẫu), rồi đo những thứ quyết định chọn công cụ trong thực tế: thời gian nạp, dung lượng đĩa, độ
trễ p50/p95, recall so với kNN chính xác, lọc theo metadata, cập nhật và xoá, mở lại từ đĩa.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell `explore` cần cache `data/raw/zalo-legal/cache/bge-m3.npy` do bài Intermediate 01 tạo (nếu chưa có, chạy
  cell embed của bài đó trước) và Ollama với `bge-m3` để embed 300 câu hỏi.
- Chấm: `uv run pytest tracks/ai-engineer/modules/12-embeddings-vector-db/exercises/intermediate/02-vector-dbs -q`

In [ ]:
# explore: tìm gốc repo, đường dẫn dữ liệu và cache
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ZALO = ensure("zalo-legal")
CACHE = ZALO / "cache"
print(sorted(p.name for p in CACHE.iterdir()) if CACHE.exists() else "chưa có cache: chạy Intermediate 01 trước")

In [ ]:
import logging
import tempfile
import time
import uuid
import warnings

import numpy as np

warnings.filterwarnings("ignore")
logging.getLogger("chromadb").setLevel(logging.ERROR)

Hit = tuple[str, float]   # (uid, cosine similarity), cao hơn là gần hơn

## 1. Mỗi công cụ trả "khoảng cách" theo một quy ước

Với vector **đơn vị** $$a, b$$ và $$c = \cos(a, b) = a^\top b$$:

| `kind` | Ai trả | Giá trị | Đổi về cosine |
| --- | --- | --- | --- |
| `"cosine_similarity"` | Qdrant (`Distance.COSINE`), FAISS `IndexFlatIP` | $$c$$ | giữ nguyên |
| `"cosine_distance"` | Chroma (`space: cosine`), LanceDB (`metric cosine`), pgvector `<=>` | $$1 - c$$ | ? |
| `"squared_l2"` | Chroma mặc định (`l2`), LanceDB mặc định, FAISS `IndexFlatL2` | $$\lVert a - b\rVert^2$$ | ? |
| `"l2"` | pgvector `<->` | $$\lVert a - b\rVert$$ | ? |

`distance_to_cosine(d, kind) -> np.ndarray`: đổi mảng `d` về cosine theo bảng (`ValueError` với `kind` lạ). Mục 3
lý thuyết có đẳng thức cần dùng.

`latency_summary(ms) -> dict`: từ list thời gian (mili giây) của từng query, trả `{"p50": ..., "p95": ..., "mean": ...}`
dùng `np.percentile` mặc định (nội suy tuyến tính), kiểu `float`.

In [ ]:
def distance_to_cosine(d, kind: str) -> np.ndarray:
    """Đổi khoảng cách/điểm của vector store về cosine similarity (vector đơn vị)."""
    # TODO
    raise NotImplementedError


def latency_summary(ms: list[float]) -> dict:
    """p50, p95 (np.percentile mặc định) và trung bình của các thời gian (ms)."""
    # TODO
    raise NotImplementedError

## 2. Giao diện chung và mẫu Chroma

Mỗi store là một lớp với cùng các phương thức:

- `__init__(dim, path=None, name="zalo")`: `path=None` là chạy trong bộ nhớ; có `path` thì lưu ra đĩa và mở lại được.
- `upsert(uids, vectors, metas)`: thêm hoặc **thay** (uid đã có thì vector và metadata mới thay cái cũ). `uids` là list
  `str` (ví dụ `"100/2019/nđ-cp+5"`), `vectors` float32 `(n, dim)` đã chuẩn hoá, `metas` list `dict` giá trị `str`.
- `search(q, k, where=None) -> list[Hit]`: tối đa `k` cặp `(uid, cosine)` giảm dần theo cosine. `where` là `None` hoặc
  một `dict` **một khoá** so sánh bằng, ví dụ `{"law": "100/2019/nđ-cp"}`: chỉ trả bản ghi có đúng giá trị đó. Có ít hơn
  `k` bản ghi khớp thì trả ít hơn, không bao giờ trả phần tử giả.
- `delete(uids)`, `count() -> int`, `close()`.

Đọc kỹ `ChromaStore` dưới đây (đã xong) trước khi viết ba lớp còn lại. Chi tiết đáng để ý: Chroma mặc định dùng
**L2**, phải khai báo `space: cosine` khi tạo collection; `add` của Chroma bỏ qua id đã có nên dùng `upsert`; một lần
gửi có giới hạn kích thước lô.

In [ ]:
class ChromaStore:
    """Mẫu: Chroma (HNSW bên trong), collection tạo với space cosine."""

    def __init__(self, dim: int, path: str | None = None, name: str = "zalo"):
        import chromadb

        self.client = chromadb.PersistentClient(path=path) if path else chromadb.EphemeralClient()
        self.col = self.client.get_or_create_collection(name, configuration={"hnsw": {"space": "cosine"}})

    def upsert(self, uids: list[str], vectors: np.ndarray, metas: list[dict], batch: int = 4096) -> None:
        for i in range(0, len(uids), batch):
            self.col.upsert(ids=list(uids[i:i + batch]), embeddings=np.asarray(vectors[i:i + batch]),
                            metadatas=list(metas[i:i + batch]))

    def search(self, q: np.ndarray, k: int, where: dict | None = None) -> list[Hit]:
        r = self.col.query(query_embeddings=[np.asarray(q, dtype=np.float32)], n_results=k, where=where)
        cos = distance_to_cosine(r["distances"][0], "cosine_distance")
        return [(uid, float(c)) for uid, c in zip(r["ids"][0], cos)]

    def delete(self, uids: list[str]) -> None:
        self.col.delete(ids=list(uids))

    def count(self) -> int:
        return self.col.count()

    def close(self) -> None:
        pass

## 3. Ba adapter của bạn

**`FaissStore`**: FAISS chỉ là thư viện index, không có id dạng chuỗi, metadata hay bộ lọc theo trường. Bạn tự giữ:
bảng `uid -> id số` (id số tăng dần, không tái sử dụng), `id số -> uid`, `id số -> metadata`. Index là
`faiss.IndexIDMap2(faiss.IndexFlatIP(dim))` (`add_with_ids`, `remove_ids` với mảng `int64`). Lọc: truyền
`params=faiss.SearchParameters(sel=faiss.IDSelectorBatch(ids_khớp))` cho `search`; khi không đủ kết quả FAISS điền
id `-1`. `upsert` một uid đã có thì xoá id cũ rồi thêm id mới. `path` được bỏ qua (FAISS lưu bằng `faiss.write_index`,
ngoài phạm vi lớp này).

**`QdrantStore`**: `QdrantClient(path=path)` hoặc `QdrantClient(":memory:")` (local mode, không cần Docker). Tạo
collection nếu chưa có, `VectorParams(size=dim, distance=Distance.COSINE)`. Id của point phải là số nguyên không âm
hoặc UUID: dùng `str(uuid.uuid5(uuid.NAMESPACE_URL, uid))` (cùng uid luôn ra cùng UUID, kể cả khi mở lại), lưu uid và
metadata trong `payload`. Lọc bằng `models.Filter(must=[models.FieldCondition(key=..., match=models.MatchValue(value=...))])`.
`count` dùng `exact=True`; `close` gọi `client.close()` (local mode khoá thư mục cho tới khi đóng).

**`LanceStore`**: `lancedb.connect(path)` (`path=None` thì tạo một thư mục tạm bằng `tempfile.mkdtemp()`). Bảng tạo
ở lần `upsert` đầu (hoặc mở lại nếu đã có tên đó) với các cột `uid`, mỗi khoá metadata một cột, và `vector`. Thêm hoặc
thay bằng `merge_insert("uid")`. Tìm bằng `search(q).distance_type("cosine")`; `where` là chuỗi SQL, ví dụ
`law = '100/2019/nđ-cp'`: giá trị chứa dấu nháy đơn phải được escape (nháy đơn viết thành hai nháy đơn), giống mọi
chuỗi SQL. `delete` nhận một predicate SQL (`uid IN (...)`).

In [ ]:
class FaissStore:
    """FAISS IndexFlatIP (exact) + tự quản lý uid, metadata, lọc bằng IDSelectorBatch."""

    def __init__(self, dim: int, path: str | None = None, name: str = "zalo"):
        import faiss

        self.faiss = faiss
        self.index = faiss.IndexIDMap2(faiss.IndexFlatIP(dim))
        self.int_of: dict[str, int] = {}
        self.uid_of: dict[int, str] = {}
        self.meta: dict[int, dict] = {}
        self.next_id = 0

    def upsert(self, uids: list[str], vectors: np.ndarray, metas: list[dict]) -> None:
        """Thêm hoặc thay; uid đã có thì xoá id cũ rồi thêm id mới."""
        # TODO
        raise NotImplementedError

    def search(self, q: np.ndarray, k: int, where: dict | None = None) -> list[Hit]:
        """Top-k (uid, cosine), lọc bằng IDSelectorBatch; bỏ id -1."""
        # TODO
        raise NotImplementedError

    def delete(self, uids: list[str]) -> None:
        """Xoá theo uid (bỏ qua uid không có)."""
        # TODO
        raise NotImplementedError

    def count(self) -> int:
        """Số bản ghi hiện có."""
        # TODO
        raise NotImplementedError

    def close(self) -> None:
        pass


class QdrantStore:
    """Qdrant local mode (không cần server), distance COSINE, id là UUID5 của uid."""

    def __init__(self, dim: int, path: str | None = None, name: str = "zalo"):
        # TODO
        raise NotImplementedError

    @staticmethod
    def point_id(uid: str) -> str:
        """UUID5 (namespace URL) của uid, dạng chuỗi."""
        # TODO
        raise NotImplementedError

    def upsert(self, uids: list[str], vectors: np.ndarray, metas: list[dict], batch: int = 1024) -> None:
        """Thêm hoặc thay; uid và metadata nằm trong payload."""
        # TODO
        raise NotImplementedError

    def search(self, q: np.ndarray, k: int, where: dict | None = None) -> list[Hit]:
        """Top-k (uid, cosine) với bộ lọc payload."""
        # TODO
        raise NotImplementedError

    def delete(self, uids: list[str]) -> None:
        """Xoá theo uid."""
        # TODO
        raise NotImplementedError

    def count(self) -> int:
        """Số point (đếm chính xác)."""
        # TODO
        raise NotImplementedError

    def close(self) -> None:
        self.client.close()


def sql_quote(value: str) -> str:
    """Chuỗi SQL có nháy đơn hai đầu, nháy đơn bên trong nhân đôi."""
    # TODO
    raise NotImplementedError


class LanceStore:
    """LanceDB (tệp Lance trên đĩa), không index: quét phẳng; metric cosine."""

    def __init__(self, dim: int, path: str | None = None, name: str = "zalo"):
        import lancedb

        self.dim, self.name = dim, name
        self.db = lancedb.connect(path or tempfile.mkdtemp())
        self.table = self.db.open_table(name) if name in self.db.table_names() else None

    def _arrow(self, uids, vectors, metas):
        import pyarrow as pa

        cols = {"uid": list(uids)}
        for key in metas[0] if len(metas) else []:
            cols[key] = [m[key] for m in metas]
        flat = pa.array(np.ascontiguousarray(vectors, dtype=np.float32).ravel())
        cols["vector"] = pa.FixedSizeListArray.from_arrays(flat, self.dim)
        return pa.table(cols)

    def upsert(self, uids: list[str], vectors: np.ndarray, metas: list[dict]) -> None:
        """Tạo bảng ở lần đầu; các lần sau merge_insert theo uid."""
        # TODO
        raise NotImplementedError

    def search(self, q: np.ndarray, k: int, where: dict | None = None) -> list[Hit]:
        """Top-k (uid, cosine), where thành SQL đã escape."""
        # TODO
        raise NotImplementedError

    def delete(self, uids: list[str]) -> None:
        """Xoá bằng predicate uid IN (...)."""
        # TODO
        raise NotImplementedError

    def count(self) -> int:
        """Số dòng hiện có."""
        # TODO
        raise NotImplementedError

    def close(self) -> None:
        pass

In [ ]:
# explore: kiểm nhanh bốn store trên dữ liệu sinh: phải trùng exact cosine, kể cả khi lọc
rng = np.random.default_rng(0)
toy = rng.normal(size=(300, 16)).astype(np.float32)
toy /= np.linalg.norm(toy, axis=1, keepdims=True)
toy_uids = [f"law{i % 4}+{i}" for i in range(300)]
toy_metas = [{"law": f"law{i % 4}"} for i in range(300)]
for cls in (ChromaStore, FaissStore, QdrantStore, LanceStore):
    store = cls(16, name=f"toy-{cls.__name__.lower()}")
    store.upsert(toy_uids, toy, toy_metas)
    hits = store.search(toy[5], 3, where={"law": "law1"})
    print(f"{cls.__name__:12s} count={store.count()} top-3 law1: {[(u, round(c, 4)) for u, c in hits]}")
    store.close()
order = [i for i in np.argsort(-(toy @ toy[5])) if i % 4 == 1][:3]
print("exact        ", [(toy_uids[i], round(float(toy[i] @ toy[5]), 4)) for i in order])

## 4. Đo trên 61 425 vector

Mọi store chạy **trên đĩa** (thư mục tạm, xoá ở cuối) để đo được dung lượng và thời gian mở lại. Metadata của mỗi
điều là `law` = số hiệu văn bản (phần trước dấu `+` của uid). Truy vấn là 300 câu hỏi ghim (300 query-id đầu của
`qrels/test.jsonl`). Ground truth là kNN chính xác bằng numpy; ngoài recall của ANN còn tính Recall@10 theo nhãn
(điều đúng có trong top-10 không, định nghĩa của module 13) để thấy recall ANN 0.99 hay 1.0 khác nhau bao nhiêu với
người dùng.

In [ ]:
# explore: nạp ma trận, câu hỏi, ground truth
import json
import shutil

import llmkit
import pandas as pd

ids_all = json.loads((CACHE / "bge-m3.ids.json").read_text(encoding="utf-8"))
matrix = np.load(CACHE / "bge-m3.npy")
metas_all = [{"law": u.rsplit("+", 1)[0]} for u in ids_all]
qrels: dict[str, set[str]] = {}
for line in (ZALO / "qrels" / "test.jsonl").open(encoding="utf-8"):
    row = json.loads(line)
    qrels.setdefault(row["query-id"], set()).add(row["corpus-id"])
qids = list(qrels)[:300]
qtext = {q["_id"]: q["text"] for q in map(json.loads, (ZALO / "queries.jsonl").open(encoding="utf-8"))}
queries = llmkit.embed([qtext[q] for q in qids])
queries /= np.linalg.norm(queries, axis=1, keepdims=True)
exact = np.argsort(-(queries @ matrix.T), axis=1, kind="stable")[:, :10]
exact_uids = [[ids_all[i] for i in row] for row in exact]
print(matrix.shape, len(qids), "câu hỏi;", len({m["law"] for m in metas_all}), "văn bản")


def dir_mb(path: str) -> float:
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / 2**20


def evaluate(store, name: str, where_fn=None) -> dict:
    found, times = [], []
    for i, q in enumerate(queries):
        t0 = time.perf_counter()
        hits = store.search(q, 10, where=where_fn(i) if where_fn else None)
        times.append(1000 * (time.perf_counter() - t0))
        found.append([u for u, _ in hits])
    ann = np.mean([len(set(f) & set(e)) / 10 for f, e in zip(found, exact_uids)])
    ir = np.mean([len(set(f) & qrels[q]) / len(qrels[q]) for f, q in zip(found, qids)])
    return {"store": name, "recall ANN@10": ann, "Recall@10 (nhãn)": ir, **latency_summary(times)}

In [ ]:
# explore: nạp từng store (lần lượt, đóng trước khi sang store sau) và đo
WORK = Path(tempfile.mkdtemp(prefix="m12-vectordb-"))
results, builds = [], []
for cls in (FaissStore, ChromaStore, QdrantStore, LanceStore):
    path = str(WORK / cls.__name__)
    t0 = time.perf_counter()
    store = cls(1024, path=path)
    store.upsert(ids_all, matrix, metas_all)
    build = time.perf_counter() - t0
    if cls is FaissStore:
        Path(path).mkdir(exist_ok=True)
        store.faiss.write_index(store.index, str(Path(path) / "index.faiss"))
    builds.append({"store": cls.__name__, "nạp (s)": build, "đĩa (MB)": dir_mb(path), "count": store.count()})
    results.append(evaluate(store, cls.__name__))
    store.close()
    print(f"{cls.__name__}: nạp {build:.1f} s", flush=True)
display(pd.DataFrame(builds).round(1).to_string(index=False))
display(pd.DataFrame(results).round(3).to_string(index=False))

In [ ]:
# explore: LanceDB có index (IVF_HNSW_SQ) so với quét phẳng; FAISS HNSW so với FaissStore exact
lance = LanceStore(1024, path=str(WORK / "LanceStore"))
t0 = time.perf_counter()
lance.table.create_index(metric="cosine", index_type="IVF_HNSW_SQ", vector_column_name="vector")
print(f"LanceDB create_index IVF_HNSW_SQ: {time.perf_counter() - t0:.1f} s, đĩa {dir_mb(str(WORK / 'LanceStore')):.0f} MB")
results.append(evaluate(lance, "LanceStore + IVF_HNSW_SQ"))

import faiss

faiss.omp_set_num_threads(4)
hnsw = faiss.IndexHNSWFlat(1024, 16, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 100
t0 = time.perf_counter()
hnsw.add(matrix)
print(f"FAISS HNSW (M=16, efC=100): dựng {time.perf_counter() - t0:.1f} s")
for ef in (16, 64):
    hnsw.hnsw.efSearch = ef
    times, found = [], []
    for q in queries:
        t0 = time.perf_counter()
        _, ids = hnsw.search(q[None], 10)
        times.append(1000 * (time.perf_counter() - t0))
        found.append([ids_all[i] for i in ids[0]])
    ann = np.mean([len(set(f) & set(e)) / 10 for f, e in zip(found, exact_uids)])
    ir = np.mean([len(set(f) & qrels[q]) / len(qrels[q]) for f, q in zip(found, qids)])
    results.append({"store": f"FAISS HNSW efSearch={ef}", "recall ANN@10": ann, "Recall@10 (nhãn)": ir,
                    **latency_summary(times)})
display(pd.DataFrame(results).round(3).to_string(index=False))

In [ ]:
# explore: lọc theo văn bản (law của điều đúng đầu tiên): cùng câu hỏi, chỉ tìm trong một văn bản
law_of = [sorted(qrels[q])[0].rsplit("+", 1)[0] for q in qids]
filtered = []
for cls in (FaissStore, ChromaStore, QdrantStore, LanceStore):
    store = cls(1024, path=str(WORK / cls.__name__))
    if cls is FaissStore:
        store.upsert(ids_all, matrix, metas_all)
    row = evaluate(store, cls.__name__ + " + where law", where_fn=lambda i: {"law": law_of[i]})
    filtered.append(row)
    store.close()
display(pd.DataFrame(filtered).drop(columns=["recall ANN@10"]).round(3).to_string(index=False))

In [ ]:
# explore: lọc SAU khi tìm (post-filter) trên FAISS HNSW: lấy top-k' rồi mới lọc, đếm câu không đủ 10 kết quả
law_arr = np.array([m["law"] for m in metas_all])
hnsw.hnsw.efSearch = 64
for k_prime in (10, 100, 1000):
    short = 0
    for i, q in enumerate(queries):
        _, ids = hnsw.search(q[None], k_prime)
        kept = [j for j in ids[0] if j >= 0 and law_arr[j] == law_of[i]][:10]
        short += len(kept) < 10
    print(f"post-filter, lấy top-{k_prime:4d} rồi lọc: {short}/300 câu trả ít hơn 10 kết quả")
sizes = pd.Series(law_arr).value_counts()
print("số điều mỗi văn bản: trung vị", int(sizes.median()), "| văn bản của 300 câu có dưới 10 điều:",
      sum(sizes[l] < 10 for l in law_of))

In [ ]:
# explore: cập nhật và xoá: xoá 1 000 điều đúng của các câu hỏi rồi tìm lại; mở lại từ đĩa
gone = sorted({u for q in qids for u in qrels[q]})[:1000]
for cls in (ChromaStore, QdrantStore, LanceStore):
    path = str(WORK / cls.__name__)
    t0 = time.perf_counter()
    store = cls(1024, path=path)
    reopen = time.perf_counter() - t0
    before = store.count()
    t0 = time.perf_counter()
    store.delete(gone)
    t_del = time.perf_counter() - t0
    leaked = sum(any(u in gone for u, _ in store.search(q, 10)) for q in queries)
    t0 = time.perf_counter()
    store.upsert(gone, matrix[[ids_all.index(u) for u in gone]], [{"law": u.rsplit("+", 1)[0]} for u in gone])
    t_up = time.perf_counter() - t0
    print(f"{cls.__name__:12s} mở lại {reopen:5.2f} s | count {before} -> xoá {len(gone)} trong {t_del:.2f} s, "
          f"câu còn thấy điều đã xoá: {leaked} | thêm lại {t_up:.2f} s, count {store.count()}")
    store.close()

In [ ]:
# explore: dọn thư mục tạm
shutil.rmtree(WORK, ignore_errors=True)

## 5. (Tuỳ chọn, không chấm) pgvector trong Docker

pgvector biến Postgres thành vector store: cột `vector(1024)`, toán tử `<=>` (cosine distance), index `hnsw` hoặc
`ivfflat`, và lọc bằng `WHERE` của SQL. Phần này cần Docker nên không có test. Dùng một container **riêng** với tên và
cổng của bạn, đừng đụng Postgres của track Data Engineer hay dự án khác đang chạy trên máy:

```bash
# thêm một dòng vào setup/.env (chọn cổng còn trống), ví dụ: PGVECTOR_PORT=15435
docker run -d --name ae-pgvector -e POSTGRES_PASSWORD=ae -p 127.0.0.1:${PGVECTOR_PORT:-15435}:5432 pgvector/pgvector:pg17
# dọn khi xong
docker rm -f ae-pgvector
```

Cell dưới tự bỏ qua nếu không kết nối được.

In [ ]:
# explore: pgvector (tự bỏ qua nếu chưa chạy container ae-pgvector)
import psycopg

from testkit import _load_env_file

_load_env_file(ROOT / "setup" / ".env")
port = int(os.environ.get("PGVECTOR_PORT", "15435"))
try:
    conn = psycopg.connect(f"postgresql://postgres:ae@127.0.0.1:{port}/postgres", autocommit=True, connect_timeout=3)
except psycopg.OperationalError as exc:
    conn = None
    print(f"bỏ qua pgvector: không kết nối được cổng {port} ({type(exc).__name__})")
if conn is not None:
    from pgvector.psycopg import register_vector

    conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
    register_vector(conn)
    conn.execute("DROP TABLE IF EXISTS zalo")
    conn.execute("CREATE TABLE zalo (uid text PRIMARY KEY, law text, embedding vector(1024))")
    t0 = time.perf_counter()
    with conn.cursor().copy("COPY zalo (uid, law, embedding) FROM STDIN WITH (FORMAT BINARY)") as copy:
        copy.set_types(["text", "text", "vector"])
        for u, m, v in zip(ids_all, metas_all, matrix):
            copy.write_row([u, m["law"], v])
    print(f"COPY 61 425 dòng: {time.perf_counter() - t0:.1f} s")
    t0 = time.perf_counter()
    conn.execute("SET maintenance_work_mem = '1GB'")
    conn.execute("CREATE INDEX ON zalo USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 100)")
    print(f"CREATE INDEX hnsw: {time.perf_counter() - t0:.1f} s")
    times, found = [], []
    for q in queries:
        t0 = time.perf_counter()
        rows = conn.execute("SELECT uid FROM zalo ORDER BY embedding <=> %s LIMIT 10", (q,)).fetchall()
        times.append(1000 * (time.perf_counter() - t0))
        found.append([r[0] for r in rows])
    ann = np.mean([len(set(f) & set(e)) / 10 for f, e in zip(found, exact_uids)])
    print("pgvector hnsw (ef_search mặc định 40): recall ANN@10", round(ann, 3), latency_summary(times))
    conn.close()

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Store nào nạp nhanh nhất, chiếm ít đĩa nhất, truy vấn nhanh nhất? Có store nào thắng ở mọi cột không?
- Recall của ANN dưới 1 làm Recall@10 theo nhãn thay đổi bao nhiêu? Con số đó so với khoảng cách giữa dense và
  hybrid ở module 13 thế nào?
- Vì sao lọc **trước** (bộ lọc nằm trong truy vấn) luôn trả đủ kết quả còn post-filter thì không, và vì sao tăng `k'`
  chỉ giảm chứ không hết vấn đề?

Sau đó làm phần trắc nghiệm của bài trong app.